Libraries

In [21]:
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from pathlib import Path

Build layer

In [23]:
from torch_geometric.nn import SAGEConv

class GraphSAGE_GRU (nn.Module):
    def __init__(self, input_dim, output_dim, num_vertex):
        super().__init__()

        self.conv = SAGEConv(in_channels=input_dim, out_channels=output_dim)
        
        self.gru = nn.GRU(input_size=output_dim, hidden_size=output_dim, num_layers=2, batch_first=True)
        self.out = nn.Linear(in_features=output_dim, out_features=num_vertex)

    def forward(self, X, edge_index):
        # X = [T, V, F]
        list = [] # [T, V, hidden_dim]
        for t in range(X.size()):
            X_time = X[t] #[V, F]
            h = self.conv(X_time, edge_index) #[V, hidden_dim]
            h = F.relu(h)
            h = F.dropout(h, p=0.2, training=self.training)
            list.append(h)

        L = torch.stack(list, dim=0) #[T, V, hidden_dim]
        L = L.permute(1, 0, 2) #[V, T, hidden_dim]
        out, h = self.gru(L) # out[V, T, num_vertex]
        out = F.relu(out)
        out = F.dropout(out, p=0.2, training=self.training)
        return out.permute(1, 0, 2) #[T, V, num_vertex]

Take a data from folder master_cp/runs of vertices

In [ ]:
root_path = Path("../master_cp/runs")
all_files = root_path.glob('*/*/vertex_features.csv')
df_list = []

for file in all_files:
    df_temp = pd.read_csv(file) 
    
    df_list.append(df_temp)

final_df = pd.concat(df_list, ignore_index=True)

print(final_df.head())

  iteration vertex_id degree normalized_degree      dual  \
0         0         0      4          0.137931       0.0   
1         0         1      4          0.137931       0.0   
2         0         2      6          0.206897      -0.0   
3         0         3      6          0.206897       0.0   
4         0         4      6          0.206897  0.666667   

  weighted_neighbor_dual complement_degree stable_set_occurrences  \
0                    0.0                25                     52   
1                    0.0                25                     52   
2                    0.0                23                     75   
3                    1.0                23                     75   
4                    2.0                23                     41   

  score_contribution  
0                0.0  
1                0.0  
2                0.0  
3                0.0  
4                1.0  


Take y_label (score_contribution) and x features

In [ ]:
y_label = final_df["score_contribution"]
time_step = final_df["iteration"][-1]
x_label = final_df.drop(columns="score_contribution")

In [30]:
X_tensor = torch.tensor(x_label.to_numpy().astype(float))
Y_tensor = torch.tensor(y_label.to_numpy().astype(float), dtype=torch.float32)
X_tensor, Y_tensor, time_step

(tensor([[  0.0000,   0.0000,   4.0000,  ...,   0.0000,  25.0000,  52.0000],
         [  0.0000,   1.0000,   4.0000,  ...,   0.0000,  25.0000,  52.0000],
         [  0.0000,   2.0000,   6.0000,  ...,   0.0000,  23.0000,  75.0000],
         ...,
         [ 54.0000, 447.0000,  92.0000,  ...,   5.3287, 357.0000,  24.0000],
         [ 54.0000, 448.0000,  60.0000,  ...,   3.1058, 389.0000,  55.0000],
         [ 54.0000, 449.0000,  64.0000,  ...,   3.9196, 385.0000,  25.0000]],
        dtype=torch.float64),
 tensor([0.0000, 0.0000, 0.0000,  ..., 0.2618, 0.0000, 0.0000]),
 54)

In [ ]:
X_tensor.size(), Y_tensor.size()

(torch.Size([2279920, 8]), torch.Size([2279920]))

Add edge_index